In [7]:
import pyvista as pv
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import yaml
# from analytical_cases import *

# Interactive widgets
import ipywidgets as widgets
from ipywidgets import interact, Layout, interactive
from IPython.display import display

# Avoid warnings for divisions by zero (e.g., r=0)
np.seterr(divide='ignore', invalid='ignore')

{'divide': 'ignore', 'over': 'warn', 'under': 'ignore', 'invalid': 'ignore'}

In [8]:
def interactive_strain_plot(filename="output/fields.vtu",
                            analytical_case="lame",
                            tol=1e-5):
    """
    Creates an interactive plot for strain analysis from a .vtu file.

    This function allows users to select a strain field, a Z-level, and a
    plotting mode to analyze the data. It correctly handles point or cell data.

    Interactive Controls:
    - Strain Field: Dropdown to select any field with "Strain" in its name.
    - Plot Mode:
        1. 'Cylindrical': Plots radial (ε_rr), tangential (ε_θθ), and shear (ε_rθ)
           strains as a function of radius (r). Compares with analytical solution.
        2. 'Rectangular': Plots Cartesian strains (ε_xx, ε_yy, ε_xy) along the
           x-axis (at y=0).
    - Z-level: Dropdown to select the Z-coordinate for the 2D slice.

    Args:
        filename (str): Path to the input .vtu file.
        analytical_case (str, optional):
            Specifies the analytical solution ('lame' or None).
        a, b, pi, po (float): Parameters for the Lamé analytical solution.
        tol (float): Tolerance for finding points/cells at a specific Z-level or axis.
    """
    print(f"Loading data from: {filename}")
    try:
        grid = pv.read(filename)
    except FileNotFoundError:
        print(f"[ERROR] File not found: {filename}")
        return

    point_fields = {k: 'point' for k in grid.point_data if "Strain" in k}
    cell_fields = {k: 'cell' for k in grid.cell_data if "Strain" in k}
    all_strain_fields = {**point_fields, **cell_fields}

    if not all_strain_fields:
        print("[ERROR] No 'Strain' field found in the VTU file.")
        return
    print(f"Found {len(all_strain_fields)} strain fields: {list(all_strain_fields.keys())}")

    def on_field_selected(strain_field_name):
        """Called when the user selects a strain field."""
        source = all_strain_fields[strain_field_name]
        print(f"\nProcessing '{strain_field_name}' (from {source} data)...")

        if source == "point":
            coords = grid.points
            e_tensor_full = grid.point_data[strain_field_name].reshape((-1, 3, 3))
        else: # 'cell'
            coords = grid.cell_centers().points
            e_tensor_full = grid.cell_data[strain_field_name].reshape((-1, 3, 3))

        unique_z_coords = np.unique(np.round(coords[:, 2], 6))
        print(f"Found {len(unique_z_coords)} unique Z-levels.")

        def update_plot(plot_mode, z_level):
            """Called when user changes plot_mode or z_level."""
            mask = np.abs(coords[:, 2] - z_level) < tol
            if not np.any(mask):
                print(f"[WARNING] No data found at z = {z_level} ± {tol}")
                return

            sliced_coords = coords[mask]
            e_tensor = e_tensor_full[mask]
            
            x, y = sliced_coords[:, 0], sliced_coords[:, 1]
            
            # --- Create a single DataFrame with all components ---
            df = pd.DataFrame({
                'x': x, 'y': y, 'r': np.sqrt(x**2 + y**2),
                'eps_xx': e_tensor[:, 0, 0], 'eps_yy': e_tensor[:, 1, 1],
                'eps_zz': e_tensor[:, 2, 2], 'eps_xy': e_tensor[:, 0, 1]
            })
            
            _create_plot(df, plot_mode, source, z_level, strain_field_name)

        interact(
            update_plot,
            plot_mode=widgets.Dropdown(
                options=['Cylindrical', 'Rectangular'], value='Cylindrical',
                description='Plot Mode:', style={'description_width': 'initial'}
            ),
            z_level=widgets.Dropdown(
                options=unique_z_coords, value=unique_z_coords[len(unique_z_coords) // 2],
                description='Select Z-level:', style={'description_width': 'initial'},
                layout=Layout(width='50%')
            )
        )

    def _create_plot(df, mode, source, z_level, field_name):
        """Helper function to generate the Matplotlib figure."""
        fig, ax = plt.subplots(figsize=(12, 8))
        
        if mode == 'Cylindrical':
            # --- Perform coordinate transformation to cylindrical ---
            theta = np.arctan2(df['y'], df['x'])
            c, s = np.cos(theta), np.sin(theta)
            c2, s2, sc = c**2, s**2, s*c
            df['eps_rr'] = df['eps_xx'] * c2 + df['eps_yy'] * s2 + 2 * df['eps_xy'] * sc
            df['eps_tt'] = df['eps_xx'] * s2 + df['eps_yy'] * c2 - 2 * df['eps_xy'] * sc
            df['eps_rt'] = (df['eps_yy'] - df['eps_xx']) * sc + df['eps_xy'] * (c2 - s2)

            # Average results at the same radial distance
            grouped = df.groupby(pd.cut(df["r"], bins=50)).mean()
            r_avg = grouped["r"].values

            # POLISH: Use rounding for more robust grouping
            # df['r_rounded'] = df['r'].round(5)
            # grouped = df.groupby('r_rounded').mean()
            # r_avg = grouped.index.values
            
            ax.plot(r_avg, grouped["eps_rr"], 'o', label=f"ε_rr (FEM, {source})")
            ax.plot(r_avg, grouped["eps_tt"], 's', label=f"ε_θθ (FEM, {source})")
            ax.plot(r_avg, grouped["eps_zz"], 'x', label=f"ε_zz (FEM, {source})")
            ax.plot(r_avg, grouped["eps_rt"], 'd', label=f"ε_rθ (FEM, {source})")
            
            if analytical_case == 'lame':
                u_r_an = radial_displacement(r_avg, a=a, b=b, pi=pi, po=po)
                eps_tt_an = u_r_an / r_avg
                eps_rr_an = radial_strain(r_avg, a=a, b=b, pi=pi, po=po)
                ax.plot(r_avg, eps_rr_an, '--', lw=2, label="ε_rr (Analytical)")
                ax.plot(r_avg, eps_tt_an, '--', lw=2, label="ε_θθ (Analytical)")
            
            ax.set_xlabel("Radius r (m)")
            ax.set_title(f"{field_name} (Cylindrical) vs. Radius at z = {z_level}")

        elif mode == 'Rectangular':
            # Filter data to be along the x-axis (where y is close to 0)
            line_df = df[np.abs(df['y']) < tol].sort_values(by='x')
            if line_df.empty:
                ax.text(0.5, 0.5, "No data found along the x-axis (y=0)", ha='center')
            else:
                ax.plot(line_df['x'], line_df['eps_xx'], 'o-', label=f"ε_xx (FEM, {source})")
                ax.plot(line_df['x'], line_df['eps_yy'], 's-', label=f"ε_yy (FEM, {source})")
                ax.plot(line_df['x'], line_df['eps_zz'], 'x-', label=f"ε_zz (FEM, {source})")
                ax.plot(line_df['x'], line_df['eps_xy'], 'd-', label=f"ε_xy (FEM, {source})")

            ax.set_xlabel("X-coordinate (m)")
            ax.set_title(f"{field_name} (Rectangular) vs. X-coordinate at z = {z_level}")
            
        ax.set_ylabel("Strain ε (-)")
        ax.grid(True, linestyle=':')
        ax.legend()
        plt.tight_layout()
        plt.show()

    interact(
        on_field_selected,
        strain_field_name=widgets.Dropdown(
            options=list(all_strain_fields.keys()),
            description='Select Strain Field:', style={'description_width': 'initial'},
            layout=Layout(width='50%')
        )
    )

interactive_strain_plot(
    filename="output/fields.vtu",
    analytical_case=None,
)

Loading data from: output/fields.vtu
Found 2 strain fields: ['Strain_fuel', 'Strain_cladding']


interactive(children=(Dropdown(description='Select Strain Field:', layout=Layout(width='50%'), options=('Strai…

In [9]:
def interactive_stress_plot(filename="output/fields.vtu",
                            analytical_case="lame",
                            a=0.02, b=0.03, pi=1e6, po=1e7,
                            geometry_file="geometry.yaml",
                            tol=1e-5):
    """
    Creates an interactive plot for stress analysis from a .vtu file.

    This function allows users to select a stress field, a Z-level, and a
    plotting mode to analyze the data. It correctly handles point or cell data.

    Interactive Controls:
    - Stress Field: Dropdown to select any field with "Stress" in its name.
    - Plot Mode:
        1. 'Cylindrical': Plots radial (σ_rr), tangential (σ_θθ), and shear (σ_rθ)
           stresses vs. radius (r). Compares with analytical solutions.
        2. 'Rectangular': Plots Cartesian stresses (σ_xx, σ_yy, σ_xy) vs. the
           x-coordinate (at y=0).
    - Z-level: Dropdown to select the Z-coordinate for the 2D slice.

    Args:
        filename (str): Path to the input .vtu file.
        analytical_case (str, optional):
            Specifies the analytical solution ('lame', 'mariotte', 'thermal', or None).
        a, b, pi, po (float): Parameters for the Lamé analytical solution.
        geometry_file (str): Path to the YAML file needed for some analytical cases.
        tol (float): Tolerance for finding points/cells at a specific Z-level or axis.
    """
    print(f"Loading data from: {filename}")
    try:
        grid = pv.read(filename)
    except FileNotFoundError:
        print(f"[ERROR] File not found: {filename}")
        return

    geometry = None
    if analytical_case == "mariotte":
        try:
            with open(geometry_file, "r") as f:
                geometry = yaml.safe_load(f)
            print(f"Successfully loaded geometry from: {geometry_file}")
        except FileNotFoundError:
            print(f"[WARNING] Geometry file '{geometry_file}' not found. 'mariotte' case will fail.")

    point_fields = {k: 'point' for k in grid.point_data if "Stress" in k}
    cell_fields = {k: 'cell' for k in grid.cell_data if "Stress" in k}
    all_stress_fields = {**point_fields, **cell_fields}

    if not all_stress_fields:
        print("[ERROR] No 'Stress' field found in the VTU file.")
        return
    print(f"Found {len(all_stress_fields)} stress fields: {list(all_stress_fields.keys())}")

    def on_field_selected(stress_field_name):
        source = all_stress_fields[stress_field_name]
        print(f"\nProcessing '{stress_field_name}' (from {source} data)...")

        if source == "point":
            coords = grid.points
            s_tensor_full = grid.point_data[stress_field_name].reshape((-1, 3, 3))
        else: # 'cell'
            coords = grid.cell_centers().points
            s_tensor_full = grid.cell_data[stress_field_name].reshape((-1, 3, 3))

        unique_z_coords = np.unique(np.round(coords[:, 2], 6))
        print(f"Found {len(unique_z_coords)} unique Z-levels.")

        def update_plot(plot_mode, z_level):
            mask = np.abs(coords[:, 2] - z_level) < tol
            if not np.any(mask):
                print(f"[WARNING] No data found at z = {z_level} ± {tol}")
                return

            sliced_coords = coords[mask]
            s_tensor = s_tensor_full[mask]
            
            x, y = sliced_coords[:, 0], sliced_coords[:, 1]
            
            df = pd.DataFrame({
                'x': x, 'y': y, 'r': np.sqrt(x**2 + y**2),
                'sigma_xx': s_tensor[:, 0, 0], 'sigma_yy': s_tensor[:, 1, 1],
                'sigma_zz': s_tensor[:, 2, 2], 'sigma_xy': s_tensor[:, 0, 1]
            })
            
            _create_plot(df, plot_mode, source, z_level, stress_field_name)

        interact(
            update_plot,
            plot_mode=widgets.Dropdown(
                options=['Cylindrical', 'Rectangular'], value='Cylindrical',
                description='Plot Mode:', style={'description_width': 'initial'}
            ),
            z_level=widgets.Dropdown(
                options=unique_z_coords, value=unique_z_coords[len(unique_z_coords) // 2],
                description='Select Z-level:', style={'description_width': 'initial'},
                layout=Layout(width='50%')
            )
        )

    def _create_plot(df, mode, source, z_level, field_name):
        fig, ax = plt.subplots(figsize=(12, 8))
        
        if mode == 'Cylindrical':
            theta = np.arctan2(df['y'], df['x'])
            c, s = np.cos(theta), np.sin(theta)
            c2, s2, sc = c**2, s**2, s*c
            df['sigma_rr'] = df['sigma_xx'] * c2 + df['sigma_yy'] * s2 + 2 * df['sigma_xy'] * sc
            df['sigma_tt'] = df['sigma_xx'] * s2 + df['sigma_yy'] * c2 - 2 * df['sigma_xy'] * sc
            df['sigma_rt'] = (df['sigma_yy'] - df['sigma_xx']) * sc + df['sigma_xy'] * (c2 - s2)
            
            df['r_rounded'] = df['r'].round(5)
            grouped = df.groupby('r_rounded').mean()
            r_avg = grouped.index.values

            ax.plot(r_avg, grouped["sigma_rr"], 'o', label=f"σ_rr (FEM, {source})")
            ax.plot(r_avg, grouped["sigma_tt"], 's', label=f"σ_θθ (FEM, {source})")
            ax.plot(r_avg, grouped["sigma_zz"], 'x', label=f"σ_zz (FEM, {source})")
            ax.plot(r_avg, grouped["sigma_rt"], 'd', label=f"σ_rθ (FEM, {source})")
            
            # --- Analytical Solutions (only for cylindrical) ---
            if analytical_case:
                sigma_rr_an, sigma_tt_an, sigma_zz_an = (np.zeros_like(r_avg) for _ in range(3))
                if analytical_case == "lame":
                    sigma_rr_an, sigma_tt_an, sigma_zz_an, _ = lame_solutions(r_avg, a=a, b=b, pi=pi, po=po)
                elif analytical_case == "mariotte" and geometry:
                    sigma_rr_an, sigma_tt_an, sigma_zz_an = mariotte_solutions(r_avg, geometry, pi=pi)
                elif analytical_case == "thermal":
                    sigma_rr_an, sigma_tt_an, sigma_zz_an = thermal_stresses(r_avg)
                
                ax.plot(r_avg, sigma_rr_an, '--', lw=2, label="σ_rr (Analytical)")
                ax.plot(r_avg, sigma_tt_an, '--', lw=2, label="σ_θθ (Analytical)")
                ax.plot(r_avg, sigma_zz_an, '--', lw=2, label="σ_zz (Analytical)")
            
            ax.set_xlabel("Radius r (m)")
            ax.set_title(f"{field_name} (Cylindrical) vs. Radius at z = {z_level}")

        elif mode == 'Rectangular':
            line_df = df[np.abs(df['y']) < tol].sort_values(by='x')
            if line_df.empty:
                ax.text(0.5, 0.5, "No data found along the x-axis (y=0)", ha='center')
            else:
                ax.plot(line_df['x'], line_df['sigma_xx'], 'o-', label=f"σ_xx (FEM, {source})")
                ax.plot(line_df['x'], line_df['sigma_yy'], 's-', label=f"σ_yy (FEM, {source})")
                ax.plot(line_df['x'], line_df['sigma_zz'], 'x-', label=f"σ_zz (FEM, {source})")
                ax.plot(line_df['x'], line_df['sigma_xy'], 'd-', label=f"σ_xy (FEM, {source})")

            ax.set_xlabel("X-coordinate (m)")
            ax.set_title(f"{field_name} (Rectangular) vs. X-coordinate at z = {z_level}")
            
        ax.set_ylabel("Stress σ (Pa)")
        ax.grid(True, linestyle=':')
        ax.legend()
        plt.tight_layout()
        plt.show()

    interact(
        on_field_selected,
        stress_field_name=widgets.Dropdown(
            options=list(all_stress_fields.keys()),
            description='Select Stress Field:', style={'description_width': 'initial'},
            layout=Layout(width='50%')
        )
    )

interactive_stress_plot(
    filename="output/fields.vtu",
    analytical_case=None,
    a=0.02,
    b=0.03,
    pi=1e6,
    po=1e7
)

Loading data from: output/fields.vtu
Found 2 stress fields: ['Stress_fuel', 'Stress_cladding']


interactive(children=(Dropdown(description='Select Stress Field:', layout=Layout(width='50%'), options=('Stres…

In [10]:
import pyvista as pv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Explicitly import ipywidgets components
try:
    from ipywidgets import interact, widgets, Layout
except ImportError:
    print("Please install ipywidgets to use this interactive function: pip install ipywidgets")
    def interact(*args, **kwargs): pass
    def widgets(*args, **kwargs): pass
    Layout = dict

# It's better to have analytical functions defined elsewhere, but for this simple case,
# we can define a placeholder or the real function here.
def thermal_cylinder_analytical(r, a, b, Ta, Tb):
    """
    Calculates the steady-state temperature for a hollow cylinder.
    Assumes 1D radial heat conduction.
    """
    # Avoid division by zero if r is exactly a or b in ln(b/a)
    if np.isclose(a, b): return np.full_like(r, Ta)
    return Ta + (Tb - Ta) * np.log(r / a) / np.log(b / a)

def interactive_temperature_plot(filename="output/fields.vtu",
                                 analytical_case=None,
                                 # IMPROVEMENT: Analytical parameters are now flexible arguments
                                 a=0.02, b=0.03, Ta=400, Tb=300,
                                 tol=1e-5):
    """
    Creates an interactive plot for temperature analysis from a .vtu file.

    This function allows users to select a Z-level and a plotting mode to
    analyze the "Temperature" field from point data.

    Interactive Controls:
    - Plot Mode:
        1. 'Cylindrical': Plots temperature vs. radius (r). Compares with an
           analytical solution for a hollow cylinder if specified.
        2. 'Rectangular': Plots temperature vs. the x-coordinate (at y=0).
    - Z-level: Dropdown to select the Z-coordinate for the 2D slice.

    Args:
        filename (str): Path to the input .vtu file.
        analytical_case (str, optional):
            Specifies the analytical solution ('cylinder' or None).
        a, b (float): Inner and outer radii for the analytical solution.
        Ta, Tb (float): Temperatures at the inner and outer radii.
        tol (float): Tolerance for finding points at a specific Z-level or axis.
    """
    print(f"Loading data from: {filename}")
    try:
        grid = pv.read(filename)
    except FileNotFoundError:
        print(f"[ERROR] File not found: {filename}")
        return

    if "Temperature" not in grid.point_data:
        print("[ERROR] 'Temperature' field not found in point data.")
        return

    points = grid.points
    T_vals = grid.point_data["Temperature"]

    unique_z_coords = np.unique(np.round(points[:, 2], 6))
    print(f"Found {len(unique_z_coords)} unique Z-levels.")

    def update_plot(plot_mode, z_level):
        """Called when user changes plot_mode or z_level."""
        mask = np.abs(points[:, 2] - z_level) < tol
        if not np.any(mask):
            print(f"[WARNING] No nodes found at z = {z_level} ± {tol}")
            return

        sliced_points = points[mask]
        sliced_T = T_vals[mask]
        
        x, y = sliced_points[:, 0], sliced_points[:, 1]
        
        df = pd.DataFrame({
            'x': x, 'y': y, 'r': np.sqrt(x**2 + y**2), 'T': sliced_T
        })
        
        _create_plot(df, plot_mode, z_level)

    def _create_plot(df, mode, z_level):
        """Helper function to generate the Matplotlib figure."""
        fig, ax = plt.subplots(figsize=(12, 8))
        
        if mode == 'Cylindrical':
            # Average results at the same radial distance
            df['r_rounded'] = df['r'].round(5)
            grouped = df.groupby('r_rounded').mean()
            r_avg = grouped.index.values
            
            ax.plot(r_avg, grouped["T"], 'o', color='red', label="Temperature (FEM)")
            
            # --- Analytical Solution (only for cylindrical) ---
            if analytical_case == 'cylinder':
                T_analytical = thermal_cylinder_analytical(r_avg, a=a, b=b, Ta=Ta, Tb=Tb)
                ax.plot(r_avg, T_analytical, '--k', lw=2, label="Analytical (Cylinder)")
            
            ax.set_xlabel("Radius r (m)")
            ax.set_title(f"Temperature (Cylindrical) vs. Radius at z = {z_level}")

        elif mode == 'Rectangular':
            # Filter data to be along the x-axis (where y is close to 0)
            line_df = df[np.abs(df['y']) < tol].sort_values(by='x')
            if line_df.empty:
                ax.text(0.5, 0.5, "No data found along the x-axis (y=0)", ha='center')
            else:
                ax.plot(line_df['x'], line_df['T'], 'o-', color='red', label="Temperature (FEM)")

            ax.set_xlabel("X-coordinate (m)")
            ax.set_title(f"Temperature (Rectangular) vs. X-coordinate at z = {z_level}")
            
        ax.set_ylabel("Temperature T (K)")
        ax.grid(True, linestyle=':')
        ax.legend()
        plt.tight_layout()
        plt.show()

    # --- Create the interactive widgets for mode and Z-level ---
    interact(
        update_plot,
        plot_mode=widgets.Dropdown(
            options=['Cylindrical', 'Rectangular'], value='Cylindrical',
            description='Plot Mode:', style={'description_width': 'initial'}
        ),
        z_level=widgets.Dropdown(
            options=unique_z_coords,
            value=unique_z_coords[len(unique_z_coords) // 2],
            description='Select Z-level:',
            style={'description_width': 'initial'},
            layout=Layout(width='50%')
        )
    )

if __name__ == '__main__':
    # Make sure to run this in a Jupyter Notebook or IPython environment
    interactive_temperature_plot(
        filename="output/fields.vtu",
        analytical_case='cylinder',
        a=0.02,
        b=0.03,
        Ta=400,
        Tb=300
    )

Loading data from: output/fields.vtu
Found 20 unique Z-levels.


interactive(children=(Dropdown(description='Plot Mode:', options=('Cylindrical', 'Rectangular'), style=Descrip…

In [11]:
def interactive_displacement_plot(filename="output/fields.vtu",
                                  analytical_case="lame",
                                  # IMPROVEMENT: Analytical parameters are now flexible arguments
                                  a=0.02, b=0.03, pi=1e6, po=1e7,
                                  tol=1e-5):
    """
    Creates an interactive plot for displacement analysis from a .vtu file.

    This function allows users to select a Z-level and a plotting mode to
    analyze the "Displacement" field from point data.

    Interactive Controls:
    - Plot Mode:
        1. 'Cylindrical': Plots radial (u_r), tangential (u_θ), and axial (u_z)
           displacements vs. radius (r). Compares with analytical solution.
        2. 'Rectangular': Plots Cartesian displacements (u_x, u_y, u_z) vs. the
           x-coordinate (at y=0).
    - Z-level: Dropdown to select the Z-coordinate for the 2D slice.

    Args:
        filename (str): Path to the input .vtu file.
        analytical_case (str, optional):
            Specifies the analytical solution ('lame' or None).
        a, b, pi, po (float): Parameters for the Lamé analytical solution.
        tol (float): Tolerance for finding points at a specific Z-level or axis.
    """
    print(f"Loading data from: {filename}")
    try:
        grid = pv.read(filename)
    except FileNotFoundError:
        print(f"[ERROR] File not found: {filename}")
        return

    if "Displacement" not in grid.point_data:
        print("[ERROR] 'Displacement' field not found in point data.")
        return

    points = grid.points
    u_full = grid.point_data["Displacement"]

    unique_z_coords = np.unique(np.round(points[:, 2], 6))
    print(f"Found {len(unique_z_coords)} unique Z-levels.")

    def update_plot(plot_mode, z_level):
        """Called when user changes plot_mode or z_level."""
        mask = np.abs(points[:, 2] - z_level) < tol
        if not np.any(mask):
            print(f"[WARNING] No nodes found at z = {z_level} ± {tol}")
            return

        sliced_points = points[mask]
        sliced_u = u_full[mask]
        
        x, y = sliced_points[:, 0], sliced_points[:, 1]
        
        df = pd.DataFrame({
            'x': x, 'y': y, 'r': np.sqrt(x**2 + y**2),
            'ux': sliced_u[:, 0], 'uy': sliced_u[:, 1], 'uz': sliced_u[:, 2]
        })
        
        _create_plot(df, plot_mode, z_level)

    def _create_plot(df, mode, z_level):
        """Helper function to generate the Matplotlib figure."""
        fig, ax = plt.subplots(figsize=(12, 8))
        
        if mode == 'Cylindrical':
            theta = np.arctan2(df['y'], df['x'])
            df['u_r'] = df['ux'] * np.cos(theta) + df['uy'] * np.sin(theta)
            df['u_theta'] = -df['ux'] * np.sin(theta) + df['uy'] * np.cos(theta)
            
            df['r_rounded'] = df['r'].round(5)
            grouped = df.groupby('r_rounded').mean()
            r_avg = grouped.index.values

            ax.plot(r_avg, grouped["u_r"], 'o', label="u_r (FEM)")
            ax.plot(r_avg, grouped["u_theta"], 's', label="u_θ (FEM)")
            ax.plot(r_avg, grouped["uz"], 'x', label="u_z (FEM)")
            
            if analytical_case == 'lame':
                u_r_an = radial_displacement(r_avg, a=a, b=b, pi=pi, po=po)
                ax.plot(r_avg, u_r_an, '--', lw=2, label="u_r (Analytical)")
            
            ax.set_xlabel("Radius r (m)")
            ax.set_title(f"Cylindrical Displacement vs. Radius at z = {z_level}")

        elif mode == 'Rectangular':
            line_df = df[np.abs(df['y']) < tol].sort_values(by='x')
            if line_df.empty:
                ax.text(0.5, 0.5, "No data found along the x-axis (y=0)", ha='center')
            else:
                ax.plot(line_df['x'], line_df['ux'], 'o-', label="u_x (FEM)")
                ax.plot(line_df['x'], line_df['uy'], 's-', label="u_y (FEM)")
                ax.plot(line_df['x'], line_df['uz'], 'x-', label="u_z (FEM)")

            ax.set_xlabel("X-coordinate (m)")
            ax.set_title(f"Cartesian Displacement vs. X-coordinate at z = {z_level}")
            
        ax.set_ylabel("Displacement (m)")
        ax.grid(True, linestyle=':')
        ax.legend()
        plt.tight_layout()
        plt.show()

    interact(
        update_plot,
        plot_mode=widgets.Dropdown(
            options=['Cylindrical', 'Rectangular'], value='Cylindrical',
            description='Plot Mode:', style={'description_width': 'initial'}
        ),
        z_level=widgets.Dropdown(
            options=unique_z_coords,
            value=unique_z_coords[len(unique_z_coords) // 2],
            description='Select Z-level:',
            style={'description_width': 'initial'},
            layout=Layout(width='50%')
        )
    )

if __name__ == '__main__':
    # Make sure to run this in a Jupyter Notebook or IPython environment
    interactive_displacement_plot(
        filename="output/fields.vtu",
        analytical_case=None,
        a=0.02,
        b=0.03,
        pi=1e6,
        po=1e7
    )

Loading data from: output/fields.vtu
Found 20 unique Z-levels.


interactive(children=(Dropdown(description='Plot Mode:', options=('Cylindrical', 'Rectangular'), style=Descrip…